# Notebook 02: Training
All adapters are 4-bit QLoRA adapters on Qwen3-8B, trained with the same LoRA and optimizer
settings on one RTX 4070 (12GB). Each is trained directly on the base model, from the self-distilled
training sets built in notebook 01:

| Adapter | Used by | Task |
| :--- | :--- | :--- |
| `adapter_monolithic` | Monolithic | $Q \rightarrow (R, A)$ |
| `adapter_thinker` | Soft Modular **and** Hard Bottleneck | $Q \rightarrow R$ |
| `adapter_soft_solver` | Soft Modular | $(Q, R) \rightarrow A$ |
| `adapter_hard_solver` | Hard Bottleneck | $R \rightarrow A$ |

Loss is computed on completion tokens only. An adapter-only checkpoint is saved every
`save_steps` (25) steps, so the Trap Test can follow trace dependence through training. Every adapter
directory gets a `training_summary.json` with steps, loss history, token counts, runtime and peak VRAM.

If a run is interrupted (kernel restart, power cut), re-running its cell resumes from the latest checkpoint.

**Run the GPU-heavy cells one at a time** and restart the kernel if memory is not released.

In [ ]:
import sys
sys.path.insert(0, "../src")

from faithfulness import config

import json
import pandas as pd
from faithfulness.training import train_adapter

# Set to a small number (e.g. 10) for a quick end-to-end dry run; None = full training
DRY_RUN_STEPS = None
MAX_STEPS = DRY_RUN_STEPS or -1

print("Base:    ", config.BASE_MODEL)
print("LoRA:    ", json.dumps(config.LORA_CONFIG))
print("Training:", json.dumps(config.TRAIN_CONFIG))
print("Max sequence length:", config.MAX_SEQ_LENGTH)

## 1. Experimental adapters
Same base model, LoRA rank/alpha, target modules, optimizer, learning rate, schedule, batch size and
seed for all four. Only the training set differs.

In [ ]:
ROLES = ["monolithic", "thinker", "soft_solver", "hard_solver"]

def train(role):
    return train_adapter(
        role=role,
        dataset_path=config.SFT_DIR / f"{role}.json",
        output_dir=config.ADAPTER_DIR / f"adapter_{role}",
        max_steps=MAX_STEPS,
    )

In [ ]:
train("monolithic")

In [ ]:
train("thinker")

In [ ]:
train("soft_solver")

In [ ]:
train("hard_solver")

## 2. Training report
Training exposure differs by role (the Solvers receive loss on the answer only, the Thinker on the trace only),
so token counts are reported next to steps and loss.

In [ ]:
rows = []
for role in ROLES:
    path = config.ADAPTER_DIR / f"adapter_{role}" / "training_summary.json"
    if not path.exists():
        print(f"missing: {path}")
        continue
    s = json.loads(path.read_text())
    rows.append({
        "adapter": role,
        "examples": s["tokens"]["examples"],
        "epochs": round(s["epochs_completed"], 2),
        "steps": s["global_steps"],
        "effective_batch": s["effective_batch_size"],
        "lr": s["training"]["learning_rate"],
        "prompt_tokens": s["tokens"]["prompt_tokens"],
        "loss_tokens": s["tokens"]["completion_tokens"],
        "mean_loss": round(s["mean_train_loss"], 4),
        "final_loss": s["final_logged_loss"],
        "minutes": round(s["runtime_seconds"] / 60, 1),
        "peak_vram_gb": s["peak_vram_gb"],
    })

report = pd.DataFrame(rows).set_index("adapter")
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
report.to_csv(config.RESULTS_DIR / "training_report.csv")
report

In [ ]:
# Loss every logging step, one column per adapter (for the thesis appendix)
curves = {}
for role in ROLES:
    path = config.ADAPTER_DIR / f"adapter_{role}" / "training_summary.json"
    if path.exists():
        history = json.loads(path.read_text())["log_history"]
        curves[role] = pd.Series({h["step"]: h["loss"] for h in history if "loss" in h})
loss_curves = pd.DataFrame(curves)
loss_curves.to_csv(config.RESULTS_DIR / "training_loss_curves.csv")
loss_curves.iloc[::max(1, len(loss_curves) // 15)]